# Clase 9: kernels no lineales y margen blando en SVM

En la clase siete descubriste la intuición de la máquina de vectores de soporte (SVM, del inglés *Support Vector Machine*): un algoritmo que busca la línea, o "calle", que separa dos clases dejando el mayor margen posible a cada lado. En la clase ocho programaste esa idea desde cero, con un margen "duro" que exige que todos los puntos queden bien clasificados, sin excepciones.

Ese planteamiento tiene dos límites evidentes, ¿verdad? Primero, muchos datasets reales no se pueden separar con una línea recta, por muy bien que coloques el hiperplano. Segundo, incluso cuando casi se pueden separar, exigir una separación perfecta te deja a merced de un solo punto ruidoso, que puede torcer todo el resultado.

Hoy resuelves ambos problemas. Primero conoces los kernels, la herramienta que te permite separar datos con formas complicadas sin cambiar ni una línea del algoritmo base. Después flexibilizas la exigencia de separación perfecta con el margen blando, controlado por un parámetro llamado `C`. Cierras la clase ajustando ambas piezas —kernel y `C`— con la clase `SVC` de scikit-learn, la misma herramienta que ya usaste antes.

Antes de nada, instala la única dependencia extra de todo el curso: CVXOPT (Convex Optimization, optimización convexa), la biblioteca que usas más adelante para ver, a un nivel básico, cómo se resuelve el problema de optimización que hay detrás de una SVM.

In [ ]:
# CVXOPT no viene preinstalada en Google Colab. La necesitas más adelante en
# este cuaderno para ver, a un nivel básico, cómo se resuelve el problema de
# optimización cuadrática que hay detrás del entrenamiento de una SVM.
# Es la única dependencia extra de todo el curso: el resto ya viene instalado.
!pip install cvxopt

## El truco del kernel: separar lo que no se puede separar con una línea recta

Imagina que tienes un puñado de canicas rojas formando un círculo, rodeado por un anillo azul. Mirando la mesa desde arriba, en dos dimensiones, ¿existe alguna línea recta capaz de separar el círculo rojo del anillo azul? No, cualquier trazo que hagas deja canicas de los dos colores a cada lado. Pero si levantas esa misma disposición y la miras desde otro ángulo, por ejemplo empujando las canicas rojas del centro un poco hacia arriba, en tres dimensiones sí puedes deslizar una lámina plana entre ambos grupos. El problema seguía siendo el mismo: solo cambiaste el punto de vista.

Esto es justo lo que hacen los kernels en SVM. Un kernel es una función que transforma tus datos, añadiendo dimensiones nuevas, hasta un espacio donde sí existe una separación lineal. La diferencia con "añadir columnas a mano" —por ejemplo, calcular `x3 = x1 * x2` e incorporarlo como dato adicional— es que hacerlo explícitamente se vuelve carísimo en cuanto el dataset crece en características. El kernel evita ese coste: calcula el resultado que obtendrías en esa dimensión superior sin llegar a construirla nunca. A esto se le llama el "truco del kernel".

¿Por qué funciona este atajo? Porque el algoritmo de SVM que programaste en la clase ocho solo necesita el producto punto entre pares de vectores, nunca sus coordenadas por separado. A esta operación también se la llama producto interno, o *inner product* en inglés: siempre toma dos vectores y devuelve un único número que mide su similitud. Un kernel calcula precisamente ese producto punto tal como quedaría en el espacio transformado, sin viajar hasta allí. No necesitas dominar la demostración matemática completa para aprovecharlo: te basta con saber que existe esta puerta trasera y que scikit-learn ya la implementa por ti.

Un ejemplo ayuda a fijar la idea. Supón que tienes dos características, `x1` y `x2`, y quieres pasar a un espacio de segundo orden que incluya `x1²`, `x2²` y `x1 * x2`. Construir esas tres columnas nuevas para cada dato exige tiempo de cálculo. Multiplicar dos vectores ya transformados de esa manera, en cambio, equivale exactamente a aplicar la fórmula `K(x, x') = (1 + x · x')^p` sobre los valores originales, sin construir nada explícitamente. El resultado es matemáticamente idéntico, pero mucho más barato de calcular.

Los kernels más usados son:

- **Lineal**: no transforma nada, es la SVM que ya conoces de las clases siete y ocho
- **Polinómico** (`poly`): añade combinaciones de potencias de las características originales, útil cuando la frontera entre clases tiene forma de curva
- **RBF** (Radial Basis Function, función de base radial): mide la similitud entre puntos según su distancia, y es capaz de dibujar fronteras muy flexibles; es el kernel por defecto en scikit-learn
- **Sigmoide**: inspirado en las redes neuronales, útil en algunos problemas de clasificación binaria

Compruébalo con un dataset que, por construcción, no admite ninguna línea recta separadora: dos anillos concéntricos, generados con `make_circles` de scikit-learn.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.svm import SVC

# Generas un dataset con dos anillos concéntricos: imposible de
# separar con una línea recta
X, y = make_circles(n_samples=200, factor=0.4, noise=0.08, random_state=42)

# Un kernel lineal intenta trazar una recta y falla
svm_lineal = SVC(kernel='linear')
svm_lineal.fit(X, y)
print("Precisión con kernel lineal:", svm_lineal.score(X, y))

# Un kernel RBF "levanta" los datos a otra dimensión y sí los separa
svm_rbf = SVC(kernel='rbf')
svm_rbf.fit(X, y)
print("Precisión con kernel RBF:", svm_rbf.score(X, y))

Ejecuta esta celda y compara las dos precisiones. El kernel lineal se queda en torno al 59%, apenas mejor que acertar al azar entre dos clases, porque ninguna recta separa un círculo interior de un anillo exterior. El kernel RBF, en cambio, llega al 100%, porque encuentra una frontera curva que rodea exactamente el círculo interior.

Puedes visualizar esa frontera curva directamente.

In [ ]:
import numpy as np

# Creamos una rejilla de puntos que cubre todo el gráfico
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300),
                      np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300))
Z = svm_rbf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k')
plt.title("SVM con kernel RBF sobre dos anillos concéntricos")
plt.show()

Al ejecutarlo ves una frontera curva, casi circular, que separa el anillo interior del exterior; algo que el kernel lineal nunca podría dibujar. Si quieres comprobarlo con otro tipo de datos no lineales, prueba `make_moons` en lugar de `make_circles` y repite el experimento con `kernel='poly'`: verás que también encuentra una frontera curva, aunque con una forma distinta a la del RBF. De hecho, ese es justo el ejercicio uno del cuaderno de práctica de esta clase.

## El margen blando: qué hacer cuando los datos no son perfectamente separables

Piensa en quien corrige exámenes con una regla de aprobado muy estricta: cualquier persona que se quede a un solo punto de la nota de corte, suspende, sin excepciones. Esa regla es sencilla, pero frágil: un único examen mal puntuado por error tuerce toda la clasificación de aprobados y suspensos. Quien corrige de forma más flexible, en cambio, acepta que algún caso límite entre en la categoría "aprobado" aunque no cumpla la regla al cien por cien, porque sabe que exigir una separación perfecta puede ser peor que aceptar unas pocas excepciones razonables.

El margen duro que programaste en la clase ocho es como esa primera regla: exige que la restricción `yi * (w · xi + b) >= 1` se cumpla para todos los puntos, sin excepción. Si un solo punto la incumple, el algoritmo ni siquiera encuentra una solución. El margen blando introduce una variable de holgura (o *slack*) por cada punto del dataset, que mide cuánto se le permite violar la restricción. La condición pasa a ser:

```
yi * (w . xi + b) >= 1 - slack_i,  con slack_i >= 0
```

Cuando `slack_i` vale cero, ese punto cumple la condición estricta de siempre. Cuanto mayor es `slack_i`, más se le permite invadir el margen o incluso caer del lado equivocado del hiperplano. El objetivo de la optimización ya no es solo minimizar la magnitud de `w` (maximizar el margen), sino también reducir la suma de todas las holguras, ponderada por un nuevo parámetro: `C`.

`C` es exactamente el dial que decide cuán estricta es esa corrección. Un valor de `C` alto penaliza mucho las violaciones y hace que el algoritmo se comporte casi como un margen duro: prioriza clasificar bien cada punto de entrenamiento, aunque eso implique una calle estrecha y un hiperplano muy sensible a casos concretos, con más riesgo de sobreajuste. Con un valor de `C` bajo, el algoritmo tolera más violaciones a cambio de un margen ancho y una frontera más suave, que generaliza mejor a datos nuevos pero puede clasificar mal algún punto de entrenamiento por el camino.

Compruébalo con un dataset que tiene algo de ruido incorporado: unas pocas etiquetas están mal asignadas a propósito, así que ninguna frontera va a clasificar perfectamente todos los puntos. Entrenas dos SVC con el mismo kernel RBF, pero con dos valores de `C` muy distintos, y comparas.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Dataset con algo de ruido: el 8% de las etiquetas se asignan mal a propósito
X_ruido, y_ruido = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.5, flip_y=0.08, random_state=7
)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_ruido, y_ruido, test_size=0.3, random_state=42
)

valores_c_demo = [0.01, 1000.0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, c in zip(axes, valores_c_demo):
    modelo_c = SVC(kernel='rbf', C=c)
    modelo_c.fit(Xr_train, yr_train)

    xx2, yy2 = np.meshgrid(
        np.linspace(X_ruido[:, 0].min() - 0.5, X_ruido[:, 0].max() + 0.5, 200),
        np.linspace(X_ruido[:, 1].min() - 0.5, X_ruido[:, 1].max() + 0.5, 200)
    )
    Z2 = modelo_c.predict(np.c_[xx2.ravel(), yy2.ravel()]).reshape(xx2.shape)
    ax.contourf(xx2, yy2, Z2, alpha=0.3)
    ax.scatter(Xr_train[:, 0], Xr_train[:, 1], c=yr_train, edgecolors='k')
    ax.set_title(f"C = {c}")

    n_sv = len(modelo_c.support_vectors_)
    print(f"C={c}: precisión en entrenamiento={modelo_c.score(Xr_train, yr_train):.3f}, "
          f"precisión en prueba={modelo_c.score(Xr_test, yr_test):.3f}, "
          f"vectores de soporte={n_sv} de {len(Xr_train)}")

plt.show()

Fíjate en los números y en las dos calles dibujadas. Con `C=0.01`, el modelo usa 192 de los 210 puntos de entrenamiento como vectores de soporte —el 91% del total— porque tolera muchas violaciones y deja un margen tan ancho que casi todos los puntos quedan dentro de él o cerca de la frontera; la precisión en entrenamiento se queda en 0,957. Con `C=1000`, el modelo se ciñe mucho más a los datos: solo 15 puntos, el 7% del total, terminan siendo vectores de soporte, y la precisión en entrenamiento sube a 0,986 porque el algoritmo se esfuerza en clasificar bien incluso los casos difíciles.

En scikit-learn siempre puedes consultar ese número tras entrenar con el atributo `support_vectors_` de tu modelo `SVC`, tal como acabas de hacer. Como referencia orientativa, si los vectores de soporte superan el 10% de las muestras totales, conviene sospechar de sobreajuste y probar con un `C` menor o con otro kernel; es justo la señal que ves con `C=0.01` en este ejemplo. No existe un valor de `C` universalmente correcto: depende de cuánto ruido tenga tu dataset y de si prefieres arriesgarte al sobreajuste o al infraajuste. La forma habitual de decidirlo es probar varios valores y comparar el resultado sobre datos de prueba, algo que retomas en la última sección de esta clase.

## Resolver la optimización con CVXOPT

En la clase ocho resolviste la optimización de la SVM a mano, con un método sencillo de prueba y descarte de valores de `w`. Ese método funciona para aprender, pero no es el que se usa en la práctica: el reto real de entrenar una SVM con margen blando y kernels es un problema de programación cuadrática (QP, del inglés *Quadratic Programming*), y existen bibliotecas especializadas para resolverlo de forma mucho más eficiente.

Una de esas bibliotecas es CVXOPT, pensada en Python para resolver problemas convexos, entre ellos los de programación cuadrática. No hace falta que implementes tú mismo el SVM con kernel completo: ya viste la idea de optimización en la clase anterior y aquí basta con entender qué papel cumple CVXOPT y cómo se usa a un nivel básico.

El corazón de CVXOPT para esta tarea es la función `cvxopt.solvers.qp()`, que resuelve un problema con esta forma general:

```
minimizar   (1/2) * x^T * P * x + q^T * x
sujeto a    G * x <= h
            A * x = b
```

Aquí `x` es el vector de incógnitas que quieres encontrar (en el caso de la SVM, los multiplicadores de Lagrange asociados a cada punto), `P` y `q` describen la función que minimizas, y `G`, `h`, `A`, `b` definen las restricciones, incluida la que limita cada multiplicador según el parámetro `C` del margen blando. El siguiente ejemplo es mínimo, solo para ver la mecánica de la función: resuelve un problema cuadrático sencillo con dos variables, sin ninguna relación directa con las SVM todavía.

In [ ]:
import numpy as np
from cvxopt import matrix, solvers

# Problema de ejemplo: minimizar 2*x1^2 + x2^2 + x1*x2,
# sujeto a x1 >= 0, x2 >= 0, x1 + x2 >= 1
P = matrix(np.array([[4.0, 1.0], [1.0, 2.0]]))
q = matrix(np.array([0.0, 0.0]))
G = matrix(np.array([[-1.0, 0.0], [0.0, -1.0], [-1.0, -1.0]]))
h = matrix(np.array([0.0, 0.0, -1.0]))

solvers.options['show_progress'] = False
solucion = solvers.qp(P, q, G, h)
print("Solución x:", np.array(solucion['x']).flatten())

Al ejecutarlo, `solucion['x']` te da los valores óptimos de `x1` y `x2`. En el caso de una SVM con kernel, `P` se construye a partir de la matriz de kernels (el resultado de aplicar `K(xi, xj)` a cada par de puntos del dataset), y las filas de `G` y `h` incorporan tanto la condición `alpha_i >= 0` del margen duro como el límite superior `alpha_i <= C` que introduce el margen blando. Una vez resuelto el problema, los multiplicadores distintos de cero señalan cuáles son los vectores de soporte, exactamente el mismo concepto que ya conoces de la clase siete.

En la práctica, cuando entrenas una SVM con `SVC` de scikit-learn, no ves nada de esto: la biblioteca resuelve este problema de programación cuadrática por dentro, con solucionadores todavía más especializados que CVXOPT, como los que implementa la biblioteca LIBSVM sobre la que se apoya scikit-learn. Conocer esta pieza te sirve para entender qué ocurre realmente al llamar a `fit()`, aunque nunca necesites escribir una línea de CVXOPT en un proyecto real.

## Ajustar los parámetros de SVM en scikit-learn

Con la teoría de kernels y margen blando ya cubierta, toca ponerla en práctica con los parámetros de `SVC`: `kernel`, `C` y `gamma`. Recuerda el dataset de cáncer de mama que ya usaste en clases anteriores para clasificar tumores como benignos o malignos.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

datos = load_breast_cancer()
X = datos.data
y = datos.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Un primer acercamiento, antes de automatizar la búsqueda, es un bucle simple que recorre varios valores de `C` y de `kernel` y anota la precisión de cada combinación.

In [ ]:
kernels = ['linear', 'rbf', 'poly']
valores_de_c = [0.1, 1, 10, 100]

for kernel in kernels:
    for c in valores_de_c:
        modelo = SVC(kernel=kernel, C=c)
        modelo.fit(X_train, y_train)
        precision = modelo.score(X_test, y_test)
        print(f"kernel={kernel:6s}  C={c:<6}  precisión={precision:.4f}")

Fíjate en que, además de `kernel` y `C`, `SVC` acepta el parámetro `gamma`, que solo tiene efecto con los kernels `rbf`, `poly` y `sigmoid`. `gamma` controla cuánta influencia tiene cada punto de entrenamiento sobre la frontera de decisión. Con un valor bajo, esa influencia llega lejos y la frontera de decisión queda suave. Con un valor alto, en cambio, la influencia se concentra cerca de cada punto y la frontera se ajusta mucho al detalle de los datos, con más riesgo de sobreajuste. Por defecto, scikit-learn calcula un valor razonable de `gamma` a partir del número de características (la opción `'scale'`), así que no siempre necesitas tocarlo a mano; solo conviene explorarlo cuando ya has fijado un kernel no lineal y quieres afinar el resultado.

Al recorrer la salida del bucle anterior, notas un patrón claro en este dataset. Con `kernel='linear'`, la precisión apenas cambia entre distintos valores de `C`: se mueve entre 0,956 y 0,965, porque los datos ya son casi separables con una recta. Con `kernel='rbf'`, en cambio, sube de forma notable, de 0,939 con `C=0.1` hasta 0,983 con `C=100`, porque la frontera curva tiene mucho más margen para ajustarse —o sobreajustarse— a los datos de entrenamiento. Con `kernel='poly'` el efecto es más irregular, entre 0,930 y 0,956 según el valor. Esa diferencia de comportamiento es justo la que capturaste en la teoría de las secciones anteriores: cuantas más dimensiones y más libertad tiene la frontera, más importa vigilar el equilibrio entre margen y violaciones.

Cuando quieres explorar varias combinaciones de parámetros de forma más ordenada, sin anidar bucles a mano, `GridSearchCV` hace el trabajo por ti, probando cada combinación con validación cruzada y quedándose con la mejor.

In [ ]:
from sklearn.model_selection import GridSearchCV

rejilla_parametros = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.001]
}

busqueda = GridSearchCV(SVC(), rejilla_parametros, cv=5)
busqueda.fit(X_train, y_train)

print("Mejores parámetros:", busqueda.best_params_)
print("Mejor precisión en validación cruzada:", busqueda.best_score_)
print("Precisión sobre el conjunto de prueba:", busqueda.score(X_test, y_test))

`GridSearchCV` entrena una SVM por cada combinación de la rejilla, evaluándola con cinco particiones de validación cruzada (`cv=5`), y guarda en `best_params_` la que obtiene mejor rendimiento medio. En este dataset, la mejor combinación resulta ser el kernel lineal con `C=100`, con una precisión media de validación cruzada cercana a 0,971 y una precisión de 0,956 sobre el conjunto de prueba. Es la forma habitual de decidir, con datos y no a ojo, qué kernel y qué valor de `C` conviene a un problema concreto.

Una vez que `GridSearchCV` te da su mejor combinación, aplica la comprobación de sobreajuste que viste en la sección del margen blando, ahora sobre el modelo ganador.

In [ ]:
mejor_modelo = busqueda.best_estimator_
n_vectores_soporte = len(mejor_modelo.support_vectors_)
n_muestras = len(X_train)

print("Vectores de soporte:", n_vectores_soporte, "de", n_muestras)
print("Porcentaje:", round(100 * n_vectores_soporte / n_muestras, 1), "%")

El resultado ronda el 8,4%, por debajo del 10% de referencia, así que no hay señales claras de sobreajuste con esta combinación de parámetros. Si ese porcentaje se dispara por encima del 10%, ya sabes qué hacer: vuelve a la rejilla, añade valores de `C` más bajos y comprueba si la precisión sobre el conjunto de prueba se mantiene mientras el modelo se vuelve más sencillo.

Por último, ten en cuenta que `SVC` es, en su origen, un clasificador binario: separa solo dos clases a la vez. El dataset de cáncer de mama tiene exactamente dos clases (benigno y maligno), así que no lo notas. Pero si trabajas con tres o más categorías, `SVC` recurre por dentro a la estrategia uno-contra-uno (OvO, del inglés *One versus One*): entrena un clasificador binario por cada par de clases y combina sus votos. Puedes cambiarlo a uno-contra-el-resto (OvR, del inglés *One versus Rest*) con el parámetro `decision_function_shape='ovr'` si tu problema tiene muchas categorías y prefieres entrenar menos clasificadores.

## Resumen: lo que te llevas de esta clase

- Un kernel transforma los datos a un espacio de más dimensiones donde sí existe una separación lineal, sin necesidad de construirlas de forma explícita; a esto se le llama el truco del kernel
- Los kernels más comunes en `SVC` son `linear`, `poly`, `rbf` (el que usa scikit-learn por defecto) y `sigmoid`, cada uno adecuado para distintas formas de frontera
- El margen blando permite que algunos puntos violen la condición de separación o queden mal clasificados, mediante variables de holgura, en lugar de exigir una separación perfecta como el margen duro de la clase ocho
- El parámetro `C` controla ese equilibrio: cuanto más alto, más penaliza las violaciones y más se estrecha la calle de separación; cuanto más bajo, más las tolera y más ancha la deja
- CVXOPT es la biblioteca que resuelve el problema de programación cuadrática subyacente al entrenamiento de una SVM con margen blando y kernels; en Google Colab se instala aparte con `!pip install cvxopt`
- En `SVC`, ajustas kernel, `C` y `gamma` con un bucle simple o con `GridSearchCV`, comparando la precisión sobre datos de prueba para elegir la mejor combinación
- El porcentaje de vectores de soporte respecto al total de muestras es una señal práctica de sobreajuste: por encima del 10%, conviene sospechar y ajustar

**Ahora te toca a ti.** Abre el cuaderno de práctica de esta clase y entrena SVC con distintos kernels y valores de `C` sobre un dataset generado con `make_moons`.

**Siglas usadas en esta clase:** SVM (*Support Vector Machine*, máquina de vectores de soporte), RBF (*Radial Basis Function*, función de base radial), QP (*Quadratic Programming*, programación cuadrática), CVXOPT (Convex Optimization, optimización convexa), OvO (*One versus One*, uno contra uno), OvR (*One versus Rest*, uno contra el resto).